# OlehAssist -  New Israeli Immigrant Assistant

An AI agent using Google Cloud Platform to help new immigrants navigate Israeli bureaucracy, which utlizes RAG, text-to-SQL, and multimodal document processing.

## Setup Required

To run this notebook, you'll need:
- A Google Cloud Platform project with Vertex AI enabled
- A Discovery Engine datastore containing immigration documentation  
- A BigQuery dataset with Ministry of Aliyah branch information

**Configuration:** Update the `PROJECT_ID` and `APP_ID` in the cell below with your own credentials.

In [ ]:
!pip install --upgrade "google-genai" google-cloud-bigquery google-cloud-discoveryengine

In [ ]:
from typing import Dict, Any, List, Optional
from google.cloud import bigquery
from google.cloud import discoveryengine_v1 as discoveryengine
from google.colab import auth, files
from IPython.display import Markdown, display
from google import genai
from google.genai import types
import pandas as pd
import json
import io
import sys
from pydantic import BaseModel

auth.authenticate_user()

# --- CONFIGURATION ---
PROJECT_ID = "your-gcp-project-id" # @param {type:"string"}
LOCATION = "global"
DATA_STORE_LOCATION = "global"
APP_ID = "your-discovery-engine-app-id" # @param {type:"string"}
# BigQuery Config
DATASET_ID = "new_immigrant" # @param {type:"string"}
Ministry_of_aliyah_branch_table = f"{PROJECT_ID}.{DATASET_ID}.ministry_of_aliyah_branch_info"

# Initialize Clients
client = genai.Client(vertexai=True, project=PROJECT_ID,location=LOCATION)
bq_client = bigquery.Client(project=PROJECT_ID)

MODEL_ID = "gemini-2.5-flash" # @param
EVAL_MODEL = "gemini-2.5-pro" # @param

In [ ]:
# function which uses RAG to search docs in data store and give reliable answers

def search_aliyah_information(query: str) -> str:
    """
    Searches the ministry of Aliyah's knowledge base (Data Store) for information to aid new immigrants to Israel in navigating bureaucracy.
    Use this for any informational questions.

    Args:
        query: The search query (e.g., "How do I sign up for health insurance?", "How do I get a passport?").
    """
    print(f"\n[Tool Call] Searching knowledge base for: '{query}'...")
    try:
        # Specify the regional endpoint
        client_options = {"api_endpoint": f"{DATA_STORE_LOCATION}-discoveryengine.googleapis.com"}
        client = discoveryengine.SearchServiceClient(client_options=client_options)

        serving_config = f"projects/{PROJECT_ID}/locations/{DATA_STORE_LOCATION}/collections/default_collection/engines/{APP_ID}/servingConfigs/default_search"

        request = discoveryengine.SearchRequest(
            serving_config=serving_config,
            query=query,
            page_size=10,
            content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
                extractive_content_spec=discoveryengine.SearchRequest.ContentSearchSpec.ExtractiveContentSpec(
                    #pick the best match
                    max_extractive_segment_count=1
                ),
                snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(
                    return_snippet=True
                )
            )
        )

        # takes specifications and does the search, storing it in a variable 'response'
        response = client.search(request)

        results_text = ""
        has_documents = False
        for result in response.results:
            has_documents = True
            if hasattr(result.document, 'derived_struct_data'):
                data = result.document.derived_struct_data
                link = data.get('link', '')

                extractive_segments = data.get('extractive_segments', [])
                if extractive_segments:
                    for segment in extractive_segments:
                        content = segment.get('content', '').strip()
                        if content:
                            results_text += f"- {content}"
                            if link:
                                results_text += f" (Source: {link})"
                            results_text += "\n"
                else:
                    snippets = data.get('snippets', [])
                    for snippet_item in snippets:
                        snippet_content = snippet_item.get('snippet', '').strip()
                        if snippet_content and snippet_content != "No snippet is available for this page.":
                            results_text += f"- {snippet_content}"
                            if link:
                                results_text += f" (Source: {link})"
                            results_text += "\n"

        if not results_text:
            if has_documents:
                print(f"Debug: Search found {len(response.results)} documents, but no useful snippets were extracted for query: '{query}'.")
                return "I could not find an answer to your question. Try rephrasing your question or being more specific."
            else:
                print(f"Debug: Search returned no documents for query: '{query}'.")
                return "No specific information found regarding your question."
        else:
            print("grounded result: ")
            return results_text

# return error message when necessary
    except Exception as e:
        print(f"Error during search_aliyah_information: {str(e)}")
        return f"Error searching knowledge base: {str(e)}"

In [ ]:
# function using text to sql in order to extract info for ministry of aliyah locations

def find_ministry_of_aliyah_branch(query: str) -> List[Dict[str, Any]]:
    """
    Executes a SQL query against BigQuery and returns the results as a list of dictionaries.
    Args:
        query: The SQL query string to execute.
    """
    print(f"\n[Tool Call] Executing BigQuery SQL...")
    try:
        query_job = bq_client.query(query)
        results = []
        for row in query_job.result():
            results.append(dict(row.items()))
        return results
    except Exception as e:
        return [{"error": str(e)}]

In [ ]:
# We run this to get the schema to put into the System Instructions

schema_query = f"""
SELECT field_path AS column_name, data_type, description
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
WHERE table_name = 'ministry_of_aliyah_branch_info'
"""
branch_schema = find_ministry_of_aliyah_branch(schema_query)

In [ ]:
# Wrappinng the functions in tools so the model can call them

functions_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration.from_callable(client=client, callable=search_aliyah_information),
        types.FunctionDeclaration.from_callable(client=client, callable=find_ministry_of_aliyah_branch),
    ]
)

In [ ]:
# General instructions

CORE_PERSONA = """
You are 'OlehAssist', a professional and empathetic AI assistant for New Immigrants (Olim) in Israel.
Your mission is to guide users through the complex Aliyah bureaucracy with clarity and patience.

LANGUAGE HANDLING (FIRST PRIORITY):
- Your first message must be: "Hello! I am your personal Aliyah assistant.
  Before we begin, what is your preferred language?"
- DETECT the user's language from their response
  (whether they say "English" or respond in that language directly)
- Once detected, respond in that language and maintain it throughout the session
- Do NOT provide advice until the user selects a language.
"""

INTENT_SELECTION = """
INTENT SELECTION (SECOND PRIORITY):
Once the language is set, present these options EXACTLY as formatted below.
You MUST use a double line break (\\n\\n) between each title and its description:

A) GENERAL INFORMATION:
(Provide details on rights, benefits, Sal Klita, health care, etc.)

B) DOCUMENT UNDERSTANDING:
(Explaining confusing forms, bills, letters, etc.)

C) FIRST STEPS & APPOINTMENTS:
(Guiding you through the essential first steps in Israel, such as setting up a phone, bank account, and making important appointments)

Format Requirements: Use capital letters for the titles and wrap the descriptions in parentheses on a new line.
"""

ROUTING_LOGIC = """
ROUTING LOGIC:

You have THREE distinct paths. Route the user based on their query type:

PATH A: GENERAL INFORMATION
Trigger when the user asks QUESTIONS about:
- Any informational query about life as a new immigrant
Examples: "How do I get Sal Klita?", "What are my rights?", "How does health insurance work?"
→ Use the 'search_aliyah_information' tool

PATH B: DOCUMENT UNDERSTANDING
Trigger when the user wants to UNDERSTAND a specific document they have:
- Any confusing paperwork they received
Examples: "I got a letter I don't understand", "Can you explain this bill?", "What is this form asking for?"

PATH C: FIRST STEPS & APPOINTMENTS
Trigger when the user wants to COMPLETE bureaucratic tasks:
Examples: "I just made aliyah, what do I need to do now", "Can you help me with first steps in Israel?"

MENU RESPONSES:
- If the user responds with "A", "a", or "Option A" → Route to PATH A (General Information)
- If the user responds with "B", "b", or "Option B" → Route to PATH B (Document Understanding)
- If the user responds with "C", "c", or "Option C" → Route to PATH C (First Steps & Appointments)

MANDATORY UI INSTRUCTION FOR PATH B:
- When the user chooses Option B or mentions they have a document, IMMEDIATELY say:
  "To upload your document, please type the word 'upload' in the chat."
- DO NOT tell them to look for icons, buttons, or attachments
- DO NOT provide document analysis until AFTER they type 'upload' and you receive the file
- If they type anything other than 'upload', gently remind them: "Please type exactly 'upload' to select your file."
"""


STYLE_AND_FLOW = """
STYLE:
- Simple, professional, and scannable (use bullet points).
- Don't overwhelm: provide the 2-3 most important points first.
- Use strategic emojis to improve clarity:
  * 📋 For document lists/checklists
  * ✅ For completed steps or requirements met
  * ⚠️ For important warnings or deadlines
  * 📞 For contact information
  * 🏛️ For government offices/ministries
  * 💰 For payment/financial information
  * 📍 For locations/addresses
- Keep emojis professional: 1-2 per response, only where they add clarity.
- End responses naturally based on context:
  * If explaining a multi-step process: "Would you like details on [specific next step]?"
  * If listing options: "Which of these applies to your situation?"
  * If providing general info: End without a forced question - let the user guide the conversation.
"""



In [ ]:
# Instuctions for info_finder path

GENERAL_INFO_PATH = """
1) GENERAL INFORMATION PATH:
   - Use the 'search_aliyah_information' tool to provide accurate data for new immigrants in Israel on all aliyah-related questions

   Common topics include:
   * Rights/benefits: Sal Klita, tax benefits, housing/rental assistance, welfare services eligibility
   * Health & Insurance: Health insurance (kupat cholim) setup, National Insurance Institute (Bituach Leumi) registration
   * Documentation: Getting a Teudat Zehut (ID card), getting an Israeli passport, driver's license conversion, foreign academic degree evaluation
   * Education: Ulpan (Hebrew classes) enrollment, registering children in Israeli schools
   * Employment: Job search assistance, employment rights for new immigrants
   * Military: Army service requirements, lone soldier support programs
   * Financial: Arnona (municipal tax) payment procedures, opening bank accounts, tax procedures
   * Religious: Religious conversion processes
   * General procedures: Timelines, step-by-step processes for various bureaucratic tasks

   - Summarize results clearly and practically, step by step
   - If the query is vague or ambiguous, ask a clarifying question before using tools
   - If the tool returns no results, acknowledge this and suggest rephrasing or offer related topics
"""

In [ ]:
#Instructions for explaining confusing documents path

DOCUMENT_EXPLAINER_PATH = """
2) DOCUMENT UNDERSTANDING PATH (Visual Analysis):
- User can upload documents and you will analyze them to help them understand and handle them
- Initial Response: Ask the user to type 'upload' to upload a file from their local computer
(if they don't type 'upload' exactly, check first that the input is not simply a mispelling of 'upload'. If it appears as a mispelling of the word 'upload',
tell them, respectfully, to try typing 'upload' again but in the exact spelling.)
- After they type 'upload', and you have received the document (bill, government letter, contract, etc.):
- IDENTIFY: State clearly what the document is (e.g., "This is an Arnona/Property Tax bill from the Jerusalem Municipality").
- KEY INFO: Extract the most relevant info (ex. 'Total Amount Due', 'Due Date', 'Consumer ID')
- EXPLAIN: Summarize the purpose of the letter in 1-2 simple sentences.
- ACTION STEPS: Tell the user exactly what to do.
    * Example: "Go to the website listed at the bottom to pay," or "Take this to the bank to set up a Hora'at Keva (standing order)."
- RISK WARNING: If the document appears legally binding, urgent, or
high-risk (e.g., court notice, legal demand, enforcement letter, fine, or debt collection),
clearly warn the user and recommend contacting the issuing authority or a qualified professional before taking action.
"""

In [ ]:
# Instuctions for first steps helper

FIRST_TASKS_PATH = f"""
3) FIRST STEPS PATH:
Your goal is to guide the user through three essential bureaucracy steps: Phone → Bank → Ministry Appointment

STEP SEQUENCING:
- ALWAYS ask about Step 1 (phone) first
- ONLY proceed to Step 2 (bank) after confirming Step 1 is complete OR the user explicitly states they already have one
- ONLY proceed to Step 3 (Ministry) after confirming Steps 1 AND 2 are complete
- If user says they've completed a later step before an earlier one, acknowledge but still verify prerequisites

STEP 1: ISRAELI PHONE PLAN
- Ask: "Have you already signed up for an Israeli phone plan?"
- If no: Explain that a local number is required for almost every other registration (including banking and government appointments). Advise them to visit a nearby phone provider to acquire a SIM card.
- If yes: Proceed to Step 2.

STEP 2: ISRAELI BANK ACCOUNT
- Ask: "Have you opened an Israeli bank account yet?"
- If no: Inform the user that they must visit a physical bank branch of their choice.
- Provide this mandatory Document Checklist:
  * Teudat Zehut
  * Passport (from country of origin)
  * NIS cash or check (for the initial deposit to activate the account)
  * US Citizens: Must provide their Social Security Number (SSN) for FATCA compliance forms.
- If yes: Proceed to Step 3.

STEP 3: MINISTRY OF ALIYAH (MISRAD HAKLITA) APPOINTMENT
- Explain Purpose: This appointment is essential to "activate your benefits," receive your initial Sal Klita (Absorption Basket) payment, and be assigned a personal Aliyah mentor.
- Document Checklist for appointment:
  * Teudat Zehut
  * All passports (original and Israeli)
  * Bank account details (from Step 2)
  * Passport pictures
  * Proof of living abroad (if applicable)
- Action A: Refer them to book an appointment online via 'myvisit.com'. Also, tell them that if they don't succeed in booking online for any reason,
you can help them locate their locate branch and will provide them the necessary details to book an appointment by phone/email.
- Action B: If the user states they cannot successfully book an appointment online, say:
  '"I will help you find the phone, email, and address for your local branch so you can contact them directly. Which city or town do you live in?"

- TOOL LOGIC (Text-to-SQL):
  * IMPORTANT: The database 'serving' column contains city names in ENGLISH (e.g., 'Tel Aviv', 'Jerusalem').
  * SPELLING CORRECTION (BEFORE TRANSLATION): Correct typos and alternative spellings
    in ANY language first, then translate to English. Meaning, if the city name is in a
    non-English language (HEBREW, Russian, French, Spanish, etc.), you must first check for typos and then translate
    it to ENGLISH before generating the SQL.
      Examples:
        - User says "ירושלים" → translate to "jerusalem" → query for "jerusalem"
        - User says "tel aviv" → already English → query for "tel aviv"
        - User says "tlv" → recognize as "tel aviv" → query for "tel aviv"
        - User says "מיתתתרר" → correct typo to "מיתר" → translate to "meitar" → query for "meitar"
  * ONLY after the user provides a city or town name, call the `find_ministry_of_aliyah_branch` tool.
  * Table Schema to use: {branch_schema}
  * SQL Generation Rule: Generate a valid BigQuery SQL query to find the contact info for that city.
  * Fuzzy Matching: Always use `LOWER(serving) LIKE '%city_name_in_english%'` to ensure you catch the city.
  * Query Template: `SELECT branch, address, email, contact FROM {Ministry_of_aliyah_branch_table} WHERE LOWER(serving) LIKE '%city_name_in_english%'`

- Fallback: If the tool returns no results, or if you cannot find a specific branch, provide the following link:
  "I could not find a specific branch for your location in my database.
  Please refer to this official list to find your closest branch: https://www.gov.il/en/government-service-branches"
"""

In [ ]:
# Combining all variables into one master instruction string

SYSTEM_INSTRUCTIONS = f"""
{CORE_PERSONA}           # 1. Who you are, language handling
{INTENT_SELECTION}       # 2. How to present the menu
{ROUTING_LOGIC}          # 3. How to route users to paths
{STYLE_AND_FLOW}         # 4. General communication style

--- SPECIFIC PATHS ---
{GENERAL_INFO_PATH}      # 6. Path A details
{DOCUMENT_EXPLAINER_PATH} # 7. Path B details
{FIRST_TASKS_PATH}       # 8. Path C details
"""

In [ ]:
# Define configuration

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTIONS,
    tools=[search_aliyah_information, find_ministry_of_aliyah_branch],
    tool_config=types.ToolConfig(
        function_calling_config=types.FunctionCallingConfig(
            mode="AUTO" # Allows the model to choose when to call the functions
        )
    )
)

In [ ]:
# chatbot

def start_chat():
    chat = client.chats.create(model=MODEL_ID, config=config)

    # Simple start with emoji
    user_input = input("🤖 OlehAssist: Hello! I am your personal aliyah assistant. What is your preferred language?\n\n👤 You: ")

    while True:
        # Safety Check
        if not user_input.strip():
            user_input = input("👤 You (Please type something): ")
            continue

        if user_input.lower() in ["quit", "exit"]:
            print("👋 Goodbye!")
            break

        message_parts = [user_input]

        # File upload handling
        if user_input.lower() == "upload":
            print("📁 Opening local file selector...")
            uploaded = files.upload()
            if uploaded:
                filename = list(uploaded.keys())[0]
                image_bytes = uploaded[filename]
                mime = "image/png" if filename.endswith('.png') else "image/jpeg"

                message_parts = [
                    "Please explain this document for me.",
                    types.Part.from_bytes(data=image_bytes, mime_type=mime)
                ]
                print(f"✅ Uploading {filename}...")
            else:
                user_input = input("\n❌ No file selected.\n👤 You: ")
                continue

        response = chat.send_message(message_parts)

        # Tool call handling loop
        while response.candidates[0].content.parts and response.candidates[0].content.parts[0].function_call:
            fc = response.candidates[0].content.parts[0].function_call
            if fc.name == "search_aliyah_information":
                tool_output = search_aliyah_information(**fc.args)
            elif fc.name == "find_ministry_of_aliyah_branch":
                tool_output = find_ministry_of_aliyah_branch(**fc.args)
            else:
                tool_output = "Unknown tool."

            response = chat.send_message(
                types.Part.from_function_response(name=fc.name, response={"content": tool_output})
            )

        # Get response text
        text = response.text if response.text else ""

        # Menu formatting
        menu_keywords = ["GENERAL INFORMATION", "DOCUMENT UNDERSTANDING", "FIRST STEPS"]

        if text and all(key in text.upper() for key in menu_keywords):
            text = (
                "I'm here to assist you with your Aliyah journey. Please select one of the following options:\n\n"
                "**A) GENERAL INFORMATION:**\n(rights, benefits, Sal Klita, health care, etc.)\n\n"
                "**B) DOCUMENT UNDERSTANDING:**\n(confusing forms, bills, letters, etc.)\n\n"
                "**C) FIRST STEPS & APPOINTMENTS:**\n(Guiding for essential first steps in Israel, such as setting up a phone, bank account, and making your Ministry of Aliyah Appointment)"
            )

        if text:
            display(Markdown(f"**🤖 OlehAssist:** {text}"))
        else:
            print("🤖 OlehAssist: I'm sorry, I didn't catch that. Could you please repeat?")

        user_input = input("\n👤 You: ")


if __name__ == "__main__":
    start_chat()


In [ ]:
# EVALUATION

In [ ]:
# Define OlehAssist categories with subtopics

CATEGORIES_AND_SUBTOPICS = """
A) GENERAL INFORMATION (intent: general_info, tool: search_aliyah_information)
Subtopics:
- Health insurance (kupat cholim) setup
- Tax benefits for new immigrants
- Housing/rental assistance programs
- Ulpan (Hebrew classes) enrollment
- Army service requirements
- Registering children in Israeli schools
- Getting an Israeli passport
- Getting a Teudat Zehut (ID card)
- Employment and job search assistance
- Lone soldier support programs
- Welfare services eligibility
- Driver's license conversion process
- Religious conversion
- Arnona (municipal tax) payment procedures
- Foreign academic degree evaluation
- National Insurance Institute (Bituach Leumi) registration

B) DOCUMENT UNDERSTANDING (intent: document, tool: None - asks user to upload)
Subtopics:
- Bills: water, electricity, arnona, gas
- Airport arrival documents
- Government letters from ministries
- Bank statements
- Rental contracts
- Ministry correspondence

C) FIRST STEPS & APPOINTMENTS (intent: first_steps, tool: find_ministry_of_aliyah_branch)
Subtopics:
- General first steps guidance after making aliyah
- Phone plan setup (Step 1)
- Bank account opening (Step 2)
- Ministry of Aliyah appointment booking (Step 3)
"""

print("✅ Categories and subtopics defined")

In [ ]:
# Prompt to generate realistic test conversations

CONVERSATION_GENERATION_PROMPT = f"""
Your goal is to create realistic test conversations for an aliyah assistant chatbot (OlehAssist) that helps new immigrants to Israel.

Generate 20 authentic conversation scenarios covering different intents and languages.

Categories and behaviors to test:
{CATEGORIES_AND_SUBTOPICS}

Schema and Instructions:

user_turns: A list of user messages (strings) that simulate a realistic conversation.
- First turn: User selects language (e.g., "English", "עברית", "Русский", "Français", "Deutsch", "Español")
- Second turn onwards: User asks question or states need (2-5 total turns)
- For first_steps conversations, include follow-up responses like "Yes" (or in their language) when agent asks about phone/bank, and a city name at the end (e.g., "Tel Aviv", "Jerusalem", "Haifa")
- Make conversations sound authentic with natural phrasing

expected_language: The language the user selected and agent should respond in
- Options: "English", "Hebrew", "Russian", "French", "German", "Spanish"

expected_intent: Which path the conversation should follow
- "general_info" if asking informational questions (taxes, benefits, procedures)
- "document" if user wants to understand a document/bill
- "first_steps" if user needs step-by-step guidance (phone, bank, appointment)

expected_tool: Which tool should be called
- "search_aliyah_information" for general_info path
- None (null) for document path (agent asks user to type 'upload')
- "find_ministry_of_aliyah_branch" for first_steps path (when user provides city)

Generation Requirements:
- Total: 20 conversations
- Language distribution: 10 in English, 10 mixed (2 for each: "Hebrew", "Russian", "French", "German", "Spanish")
- Intent distribution:
  * 8 general_info (covering different subtopics)
  * 6 document (different types of bills/letters)
  * 6 first_steps
- Make conversations realistic and detailed
- Vary the subtopics to get good coverage
"""

print("✅ Generation prompt created")


In [ ]:
# data structure for elements in the test conversations

class ConversationTest(BaseModel):
    user_turns: List[str]
    expected_language: str
    expected_intent: str
    expected_tool: Optional[str]

# Container for all test conversations
class ConversationTestSet(BaseModel):
    conversations: List[ConversationTest]

print("✅ schemas defined")

In [ ]:
# function for Gemini to create the synthetic convresations

def generate_test_conversations() -> pd.DataFrame:
    """
    Uses EVAL_MODEL to generate synthetic test conversations
    Returns a DataFrame with test_id and expected labels
    """
    print("\n🔄 Generating synthetic test conversations with LLM...")

    response = client.models.generate_content(
        model=EVAL_MODEL,  # Use stronger model for generation
        contents=CONVERSATION_GENERATION_PROMPT,
        config=types.GenerateContentConfig(
            system_instruction=CONVERSATION_GENERATION_PROMPT,
            response_mime_type="application/json",
            response_schema=ConversationTestSet,
        )
    )

    # Parse JSON response
    data = json.loads(response.text)

    # Convert to DataFrame
    df = pd.DataFrame(data["conversations"])
    df['test_id'] = range(1, len(df) + 1)

    print(f"✅ Generated {len(df)} test conversations")

    return df

print("✅ Generation function defined")


In [ ]:
# Generate the golden set
golden_set = generate_test_conversations()

# Display summary
print("\n" + "="*80)
print("GOLDEN SET CREATED")
print("="*80)
print(f"\nTotal test cases: {len(golden_set)}")
print(f"\nBreakdown by intent:")
print(golden_set['expected_intent'].value_counts())
print(f"\nBreakdown by language:")
print(golden_set['expected_language'].value_counts())

# Show the full dataset
print("\n" + "="*80)
print("FULL GOLDEN SET:")
print("="*80)
display(golden_set[['test_id', 'expected_language', 'expected_intent', 'expected_tool']])


In [ ]:
print("\n" + "="*80)
print("STEP 3: RUNNING AGENT CONVERSATIONS")
print("="*80)

conversation_data = []

for idx, row in golden_set.iterrows():
    print(f"\n{'='*80}")
    print(f"RUNNING TEST {row['test_id']}/{len(golden_set)}: {row['expected_language']} - {row['expected_intent']}")
    print(f"{'='*80}\n")

    # Create fresh chat for this test
    chat = client.chats.create(model=MODEL_ID, config=config)

    # Start building transcript
    transcript_lines = []

    # Agent always greets first (per your system instructions)
    agent_greeting = "Hello! I am OlehAssist, your personal Aliyah assistant. Before we begin, what is your preferred language?"
    print(f"🤖 Agent (turn 0): {agent_greeting}\n")
    transcript_lines.append(f"Agent: {agent_greeting}")

    # Send each user turn from the golden set
    for turn_num, user_msg in enumerate(row['user_turns'], 1):
        print(f"👤 User (turn {turn_num}): {user_msg}")
        transcript_lines.append(f"User: {user_msg}")

        # Capture stdout to catch [Tool Call] prints
        old_stdout = sys.stdout
        sys.stdout = captured_output = io.StringIO()

        # Send message to agent
        response = chat.send_message(user_msg)

        # Restore stdout and get captured output
        sys.stdout = old_stdout
        tool_prints = captured_output.getvalue()

        # Add tool call prints to transcript
        if "[Tool Call]" in tool_prints:
            transcript_lines.append(tool_prints.strip())
            print(tool_prints.strip())

        # Add agent response to transcript
        if response.text:
            print(f"🤖 Agent (turn {turn_num}): {response.text[:150]}...\n")
            # Save FULL response (not truncated)
            transcript_lines.append(f"Agent: {response.text}")

    # Join all lines into full transcript
    full_transcript = "\n\n".join(transcript_lines)

    # Store transcript with test_id
    conversation_data.append({
        'test_id': row['test_id'],
        'transcript': full_transcript
    })

    print(f"✅ Test {row['test_id']} completed\n")

print("="*80)
print("ALL CONVERSATIONS COMPLETED")
print("="*80)


In [ ]:
# Create DataFrame from conversation data

transcript_df = pd.DataFrame(conversation_data)

# Merge transcripts with golden_set using test_id
golden_set = golden_set.merge(transcript_df, on='test_id')

print(f"\n✅ Transcripts added to golden_set")
print(f"Golden set now has {len(golden_set.columns)} columns: {list(golden_set.columns)}")
print(f"\nReady for evaluation!")


In [ ]:
# Instructions for the LLM evaluator (the "judge")
EVALUATION_INSTRUCTIONS = """
You are evaluating conversations with an immigration assistant chatbot (OlehAssist).

Read the full conversation transcript and determine the following:

1. detected_language: What language did the AGENT respond in (after user selected it)?
   - Look at the agent's responses (not user messages)
   - Identify from actual text:
     * "English" - Latin alphabet, English words
     * "Hebrew" - Hebrew characters (עברית)
     * "Russian" - Cyrillic characters (Русский)
     * "French" - French words/phrases
     * "German" - German words/phrases
     * "Spanish" - Spanish words/phrases

2. detected_intent: Which conversational path did the agent follow?
   - "general_info" if agent used search_aliyah_information tool OR provided informational answers about rights, benefits, procedures
   - "document" if agent asked user to type "upload" to analyze a document
   - "first_steps" if agent asked sequential questions about phone plan, bank account, and ministry appointment

3. tool_used: Which tool was called during the conversation?
   - Look for these EXACT phrases in the transcript:
     * "[Tool Call] Searching knowledge base" → return "search_aliyah_information"
     * "[Tool Call] Executing BigQuery SQL" → return "find_ministry_of_aliyah_branch"
     * If no tool call phrase found → return null
   - Only return one tool (the last one called if multiple)

Be precise and base your answers only on what appears in the transcript.
"""

print("✅ Evaluation instructions defined")


In [ ]:
# Pydantic schema for structured evaluation output
class ConversationEvaluation(BaseModel):
    detected_language: str
    detected_intent: str
    tool_used: Optional[str]

print("✅ Evaluation schema defined")


In [ ]:
# the judge
def evaluate_conversation_transcript(transcript: str) -> dict:
    """
    Uses EVAL_MODEL (gemini-2.5-pro) to analyze a conversation transcript
    and return structured evaluation results.

    Args:
        transcript: Full conversation text with Agent/User turns

    Returns:
        Dictionary with detected_language, detected_intent, tool_used
    """

    EVAL_PROMPT = f"""
Analyze this conversation transcript:

{transcript}

Determine the detected_language, detected_intent, and tool_used according to the instructions.
"""

    response = client.models.generate_content(
        model=EVAL_MODEL,  # Use gemini-2.5-pro for evaluation
        contents=EVAL_PROMPT,
        config=types.GenerateContentConfig(
            system_instruction=EVALUATION_INSTRUCTIONS,
            response_mime_type="application/json",
            response_schema=ConversationEvaluation,
        )
    )

    # Parse and return structured JSON
    return json.loads(response.text)

print("✅ Evaluation function created")
print(f"   Using model: {EVAL_MODEL}")


In [ ]:
print("\n" + "="*80)
print("STEP 5: EVALUATING TRANSCRIPTS WITH LLM")
print("="*80)
print(f"Using {EVAL_MODEL} to evaluate {len(golden_set)} conversations...\n")

predicted_results = []

for idx, row in golden_set.iterrows():
    print(f"🔍 Evaluating Test {row['test_id']}/{len(golden_set)}... ", end='')

    # Call evaluation function with the transcript
    eval_result = evaluate_conversation_transcript(row['transcript'])

    print(f"✓")
    print(f"   Language: {eval_result['detected_language']}")
    print(f"   Intent: {eval_result['detected_intent']}")
    print(f"   Tool: {eval_result['tool_used']}\n")

    # Store predictions
    predicted_results.append({
        'test_id': row['test_id'],
        'generated_language': eval_result['detected_language'],
        'generated_intent': eval_result['detected_intent'],
        'generated_tool': eval_result['tool_used']
    })

print("="*80)
print("✅ EVALUATION COMPLETE")
print("="*80)


In [ ]:
# Create DataFrame from predictions
predictions_df = pd.DataFrame(predicted_results)

# Merge predictions with golden_set using test_id
golden_set = golden_set.merge(predictions_df, on='test_id')

print(f"\n✅ Predictions added to golden_set")
print(f"\nColumns now in golden_set:")
print(list(golden_set.columns))

In [ ]:
print("\n" + "="*80)
print("STEP 6: CALCULATING ACCURACY")
print("="*80)

# Compare expected vs generated for each metric
golden_set['language_correct'] = (
    golden_set['expected_language'] == golden_set['generated_language']
)

golden_set['intent_correct'] = (
    golden_set['expected_intent'] == golden_set['generated_intent']
)

# Handle None == None as True for tool comparison
def tools_match(row):
    expected = row['expected_tool']
    generated = row['generated_tool']
    # Both None or both same value = correct
    if expected is None and generated is None:
        return True
    return expected == generated

golden_set['tool_correct'] = golden_set.apply(tools_match, axis=1)

# Calculate accuracy scores
language_accuracy = golden_set['language_correct'].mean()
intent_accuracy = golden_set['intent_correct'].mean()
tool_accuracy = golden_set['tool_correct'].mean()

print("\n" + "="*80)
print("EVALUATION RESULTS")
print("="*80)

print(f"\nLanguage Detection Accuracy: {language_accuracy:.2f}")
print(f"Intent Routing Accuracy:     {intent_accuracy:.2f}")
print(f"Tool Selection Accuracy:     {tool_accuracy:.2f}")

print(f"\n📊 Summary:")
print(f"   Language: {golden_set['language_correct'].sum()}/{len(golden_set)} correct")
print(f"   Intent:   {golden_set['intent_correct'].sum()}/{len(golden_set)} correct")
print(f"   Tool:     {golden_set['tool_correct'].sum()}/{len(golden_set)} correct")


In [ ]:
print("\n" + "="*80)
print("DETAILED COMPARISON TABLE")
print("="*80 + "\n")

# Select columns for comparison
comparison_df = golden_set[[
    'test_id',
    'expected_language', 'generated_language', 'language_correct',
    'expected_intent', 'generated_intent', 'intent_correct',
    'expected_tool', 'generated_tool', 'tool_correct'
]]

# Display as nice table
display(comparison_df)


In [ ]:
# Identify which tests failed
print("\n" + "="*80)
print("FAILURE ANALYSIS")
print("="*80)

# Find failures
language_failures = golden_set[golden_set['language_correct'] == False]
intent_failures = golden_set[golden_set['intent_correct'] == False]
tool_failures = golden_set[golden_set['tool_correct'] == False]

if len(language_failures) > 0:
    print(f"\n⚠️  Language Detection Failures ({len(language_failures)}):")
    for idx, row in language_failures.iterrows():
        print(f"   Test {row['test_id']}: Expected {row['expected_language']}, Got {row['generated_language']}")
else:
    print("\n✅ No language detection failures")

if len(intent_failures) > 0:
    print(f"\n⚠️  Intent Routing Failures ({len(intent_failures)}):")
    for idx, row in intent_failures.iterrows():
        print(f"   Test {row['test_id']}: Expected {row['expected_intent']}, Got {row['generated_intent']}")
else:
    print("\n✅ No intent routing failures")

if len(tool_failures) > 0:
    print(f"\n⚠️  Tool Selection Failures ({len(tool_failures)}):")
    for idx, row in tool_failures.iterrows():
        print(f"   Test {row['test_id']}: Expected {row['expected_tool']}, Got {row['generated_tool']}")
else:
    print("\n✅ No tool selection failures")